# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, lower, regexp_replace
from pyspark.sql.types import StringType


# Read from bronze table

In [0]:
df = spark.table("bike_data.bronze.loc_a101_raw")
df.display()

# Transformations

## Trimming

In [0]:
for item in df.schema.fields:
    if isinstance (item.dataType, StringType):
        df = df.withColumn(item.name, trim(col(item.name)))

## Normalization

In [0]:
df = (
    df.withColumn("CID", lower(col("CID")))
    .withColumn("CNTRY", lower(col("CNTRY")))
)

In [0]:
df = (
    df
    .withColumn("cntry",
                F.when(col("cntry") == "united kingdom", "uk")
                .when(col("cntry") == "de", "germany")
                .when(col("cntry") == "usa", "us")
                .when(col("cntry") == "united states", "us")
                .otherwise(col("cntry")))
)

In [0]:
df.display()

In [0]:
%m

### Customer Key Normalization

In [0]:
# Remove hyphen fom ERP Customer Identifier
df = (
  df.
  withColumn("cid", F.regexp_replace(col("cid"),"-",""))
)

In [0]:
df.display()

## Rename Column Names

In [0]:
df = (
    df
    .withColumnRenamed("CID", "customer_key")
    .withColumnRenamed("CNTRY", "country")
)

# Write to Silver Table

In [0]:
(
  df.write
  .mode("overwrite")
  .saveAsTable("bike_data.silver.erp_customer_location")
)